Import Libraries and Configure Logging

In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
from gensim.models.phrases import Phrases, Phraser
from sklearn.feature_extraction.text import TfidfVectorizer
from rake_nltk import Rake, Metric
from collections import Counter
from tqdm import tqdm
import logging
import random
import os
from scipy.stats import entropy

# Setup logging
logging.basicConfig(
    filename='lda_output.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logging.getLogger('gensim').setLevel(logging.WARNING)

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

Initialize NLP Tools and Stopwords

In [2]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Domain-specific stopwords
custom_stopwords = {
    'allow', 'class', 'available', 'part', 'case', 'lead', 'shall', 'product', 'operate',
    'operational', 'result', 'input', 'dependent', 'preference', 'item', 'without', 'let',
    'returned', 'message', 'every', 'system', 'run', 'fully', 'major', 'reasonable',
    'software', 'user', 'able', 'ability', 'support', 'year', 'expected', 'must',
    'information', 'data', 'use', 'using', 'provide', 'successfully', 'one', 'waiter',
    'include', 'accommodate', 'event', 'technique', 'recent', 'administrator', 'search',
    'add', 'allows', 'achieve', 'way', 'outside', 'release', 'launch', 'allowed', 'entered',
    'within', 'first', 'new', 'izogn', 'wcs', 'course', 'time', 'help', 'learn', 'ccr', 'cma'
}
stop_words.update(custom_stopwords)

# Review-specific stopwords
review_stopwords = {
    'review', 'star', 'rating', 'good', 'great', 'course', 'learn', 'learning',
    'would', 'like', 'could', 'one', 'bit', 'week', 'think', 'much', 'really',
    'lot', 'new', 'thank', 'thanks', 'many', 'well', 'also', 'get', 'time',
    'truly', 'even', 'make', 'see', 'content', 'material', 'class', 'work',
    'way', 'understand', 'information', 'helpful', 'useful', 'knowledge',
    'day', 'help', 'easy'
}
stop_words.update(review_stopwords)

Load PROMISE Data

In [3]:
def load_promise_data(file_path):
    try:
        df = pd.read_csv(file_path)
        logging.info(f"Successfully loaded PROMISE data from {file_path}")
        return df
    except FileNotFoundError:
        logging.error(f"PROMISE data file not found at {file_path}")
        raise
    except Exception as e:
        logging.error(f"Error loading PROMISE data: {e}")
        raise

promise_data_path = '../../datasets/PROMISE_exp_cleaned.csv'
promise_df = load_promise_data(promise_data_path)

Aggregate Data by Category

In [4]:
logging.info("Started aggregating PROMISE data by category")
category_texts = promise_df.groupby('_class_')['cleaned_text'].apply(lambda x: ' '.join(x)).to_dict()
for category, text in category_texts.items():
    word_count = len(text.split())
    logging.info(f"Category {category}: {word_count} words")
    print(f"Category {category}: {word_count} words")
    if word_count < 50:
        logging.warning(f"Category {category} has short Ging warning")
        print(f"Warning: Category {category} has short text ({word_count} words). Consider reviewing data.")
    if word_count == 0:
        logging.error(f"Category {category} has empty text")
        raise ValueError(f"Category {category} has empty text. Check data filtering.")

Category F: 3041 words
Category FT: 129 words
Category PE: 559 words
Category PO: 58 words
Category SC: 144 words
Category SE: 951 words
Category US: 653 words


Extract Seed Words

In [5]:
logging.info("Started extracting seed words")
seed_words = {}
target_word_count = 20
category_word_scores = {}

for category, text in category_texts.items():
    vectorizer = TfidfVectorizer(
        max_features=200,
        ngram_range=(1, 3),
        stop_words=list(stop_words),
        min_df=1,
        sublinear_tf=True
    )
    tfidf_matrix = vectorizer.fit_transform([text])
    tfidf_terms = vectorizer.get_feature_names_out()
    tfidf_scores = tfidf_matrix.toarray()[0]
    tfidf_term_scores = {term: score for term, score in zip(tfidf_terms, tfidf_scores) if score > 0.05}

    rake = Rake(
        stopwords=stop_words,
        min_length=2,
        max_length=3,
        ranking_metric=Metric.DEGREE_TO_FREQUENCY_RATIO,
        include_repeated_phrases=False
    )
    rake.extract_keywords_from_text(text)
    rake_phrases = rake.get_ranked_phrases_with_scores()[:60]

    rake_terms = []
    for score, phrase in rake_phrases:
        words = [lemmatizer.lemmatize(word) for word in phrase.split()]
        rake_terms.extend(words if len(words) == 1 else ['_'.join(words)])

    combined_terms = list(set(tfidf_terms) | set(rake_terms))
    filtered_terms = [
        term for term in combined_terms
        if (term.isalpha() or '_' in term)
        and term not in stop_words
        and len(term.replace('_', '')) > 2
        and (('_' in term) or tfidf_term_scores.get(term, 0) > 0.06)
    ]

    term_scores = {}
    word_freq = Counter(text.split())
    total_freq = sum(word_freq.values())
    for term in filtered_terms:
        term_scores[term] = tfidf_term_scores.get(term, word_freq.get(term, 1) / total_freq)

    sorted_terms = sorted(term_scores.items(), key=lambda x: x[1], reverse=True)[:target_word_count]
    seed_words[category] = [term for term, _ in sorted_terms]
    category_word_scores[category] = term_scores

Deduplicate and Finalize Seed Words

In [6]:
logging.info("Started deduplicating seed words")
final_seed_words = {category: [] for category in seed_words}
word_to_category = {}
word_to_score = {}

all_terms = []
for category, words in seed_words.items():
    for word in words:
        score = category_word_scores[category].get(word, 0)
        all_terms.append((word, category, score))

all_terms.sort(key=lambda x: x[2], reverse=True)

for word, category, score in all_terms:
    if word not in word_to_category:
        final_seed_words[category].append(word)
        word_to_category[word] = category
        word_to_score[word] = score

for category in final_seed_words:
    current_words = final_seed_words[category]
    remaining_slots = target_word_count - len(current_words)
    if remaining_slots > 0:
        available_terms = [
            (term, score) for term, score in sorted(category_word_scores[category].items(), key=lambda x: x[1], reverse=True)
            if term not in word_to_category and score > 0.04
        ]
        final_seed_words[category].extend(term for term, _ in available_terms[:remaining_slots])
    final_seed_words[category] = final_seed_words[category][:target_word_count]

Validate and Save Seed Words

In [7]:
logging.info("Validating seed words")
print("\nSeed Words by Category:")
for category, words in final_seed_words.items():
    logging.info(f"Category: {category} -> Seed Words ({len(words)}): {words}")
    print(f"Category: {category} -> Seed Words ({len(words)}): {words}")

word_overlap = {}
for category, words in final_seed_words.items():
    for word in words:
        if word not in word_overlap:
            word_overlap[word] = []
        word_overlap[word].append(category)
overlap_words = {word: cats for word, cats in word_overlap.items() if len(cats) > 1}
if overlap_words:
    logging.warning("Overlapping words detected")
    print("\nWarning: Overlapping words detected:")
    for word, cats in overlap_words.items():
        logging.warning(f"Word '{word}' appears in {cats}")
        print(f"Word '{word}' appears in {cats}")
else:
    logging.info("No overlapping words detected")
    print("\nNo overlapping words detected.")

seed_words_df = pd.DataFrame([(cat, word) for cat, words in final_seed_words.items() for word in words], columns=['Category', 'SeedWord'])
seed_words_path = '../../datasets/seed_words.csv'
seed_words_df.to_csv(seed_words_path, index=False)
logging.info(f"Seed words saved to {seed_words_path}")
print(f"\nSeed words saved to {seed_words_path}")


Seed Words by Category:
Category: F -> Seed Words (20): ['player', 'display', 'meeting', 'dispute', 'program', 'clinical', 'member', 'staff', 'student', 'lab', 'view', 'list', 'site', 'game', 'cohort', 'order', 'section', 'date', 'location', 'change']
Category: FT -> Seed Words (20): ['fault', 'failure', 'database', 'reliability', 'robust', 'continue', 'malicious', 'tablet', 'crash', 'loss', 'tolerance', 'filesystems', 'whenever', 'prevent', 'timely', 'saved', 'restored', 'payment', 'depends', 'pillar']
Category: PE -> Seed Words (20): ['second', 'response', 'minute', 'longer', 'take', 'load', 'website', 'application', 'connection', 'movie', 'fast', 'complete', 'performance', 'task', 'processing', 'card', 'pin', 'repair', 'account', 'balancing']
Category: PO -> Seed Words (20): ['web', 'window', 'operating', 'environment', 'portable', 'compatible', 'platform', 'browser', 'hardware', 'android', 'based', 'end', 'various', 'level', 'mobile', 'independence', 'function', 'wide', 'goal', 'u

Load Seed Words

In [8]:
def load_seed_words_from_csv(file_path):
    try:
        df = pd.read_csv(file_path)
        seed_words = df.groupby('Category')['SeedWord'].apply(list).to_dict()
        logging.info(f"Successfully loaded seed words from {file_path}")
        return seed_words
    except FileNotFoundError:
        logging.error(f"Seed words file not found at {file_path}")
        raise
    except Exception as e:
        logging.error(f"Error loading seed words: {e}")
        raise

seed_words = load_seed_words_from_csv(seed_words_path)
seed_word_set = set(word for words in seed_words.values() for word in words)
stop_words = stop_words - seed_word_set

Define Preprocessing Function

In [9]:
def preprocess(text):
    if not isinstance(text, str) or not text.strip():
        return []
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens
              if word not in stop_words and len(word) > 2]
    return tokens

Load and Preprocess Review Data

In [10]:
input_path = '../../datasets/review_train.csv'
output_path = '../../datasets/selected_pseudo_labeled.csv'
threshold = 0.8
entropy_threshold = 1.0  # For filtering noisy samples

os.makedirs('models', exist_ok=True)

logging.info("Started loading review data")
df = pd.read_csv(input_path)
df = df[['processed_reviews']].copy()
df['processed_reviews'] = df['processed_reviews'].fillna("")
logging.info("Completed loading review data")

logging.info("Started preprocessing reviews")
tqdm.pandas()
tokenized_reviews = df['processed_reviews'].progress_apply(preprocess)
valid_indices = [i for i, tokens in enumerate(tokenized_reviews) if tokens]
tokenized_reviews = [tokens for tokens in tokenized_reviews if tokens]
if not tokenized_reviews:
    raise ValueError("No valid reviews after preprocessing.")
logging.info("Completed preprocessing reviews")
filtered_df = df.iloc[valid_indices].copy()
logging.info(f"Filtered DataFrame to {len(filtered_df)} valid reviews")

100%|██████████| 286518/286518 [00:09<00:00, 30765.51it/s]


Detect Bigrams

In [11]:
logging.info("Started detecting bigrams")
sample_size = min(100000, len(tokenized_reviews))
sampled_reviews = random.sample(tokenized_reviews, sample_size) if sample_size < len(tokenized_reviews) else tokenized_reviews
bigram_model = Phrases(sampled_reviews, min_count=15, threshold=0.7, scoring='npmi')
bigram_phraser = Phraser(bigram_model)
logging.info("Started applying bigrams")
tokenized_reviews = [bigram_phraser[tokens] for tokens in tokenized_reviews]
logging.info("Completed applying bigrams")

Create Dictionary and Check Seed Words

In [12]:
logging.info("Started creating dictionary")
dictionary = corpora.Dictionary(tokenized_reviews)
dictionary.filter_extremes(no_below=1, no_above=0.8)
logging.info("Completed creating dictionary")

logging.info("Checking seed word presence in corpus")
all_seed_words = set(word for words in seed_words.values() for word in words)
corpus_words = set(word for review in tokenized_reviews for word in review)
missing_in_corpus = all_seed_words - corpus_words

if missing_in_corpus:
    print(f"WARNING: {len(missing_in_corpus)} seed words are NOT present in the corpus:")
    print("Missing seed words and suggested near-matches:")
    for word in sorted(list(missing_in_corpus))[:20]:
        lemmatized_word = lemmatizer.lemmatize(word)
        near_matches = [w for w in corpus_words if lemmatized_word in w or w in lemmatized_word]
        print(f"  - '{word}': Near-matches = {near_matches[:5] if near_matches else 'None'}")
    if len(missing_in_corpus) > 20:
        print(f"... and {len(missing_in_corpus) - 20} more.")
    logging.warning(f"{len(missing_in_corpus)} seed words not found in corpus")
else:
    print("SUCCESS: All seed words are present in the corpus.")
print("--------------------------------------------------\n")

print("\n--- Checking Seed Word Survival in Dictionary ---")
vocabulary = set(dictionary.token2id.keys())
missing_words = all_seed_words - vocabulary
if missing_words:
    print(f"WARNING: {len(missing_words)} seed words are NOT in the dictionary after filtering and will be ignored:")
    print(sorted(list(missing_words))[:20])
    if len(missing_words) > 20:
        print(f"... and {len(missing_words) - 20} more.")
else:
    print("SUCCESS: All seed words survived the dictionary filtering process.")
print("--------------------------------------------------\n")

Missing seed words and suggested near-matches:
  - 'database': Near-matches = ['as', 'aba', 'tab', 'da', 'base']
  - 'depends': Near-matches = ['pe', 'end', 'pen', 'depend']
  - 'encrypted': Near-matches = ['cry', 'ted']
  - 'filesystems': Near-matches = ['file', 'es', 'stem']
  - 'internet': Near-matches = ['intern', 'net', 'tern', 'terne', 'inter']
  - 'processing': Near-matches = ['sin', 'roc', 'si', 'process', 'es']
  - 'restored': Near-matches = ['rest', 'red', 'restore', 'store', 'es']
  - 'scheduling': Near-matches = ['ling', 'che', 'lin', 'li', 'ing']
  - 'secured': Near-matches = ['ure', 'unsecured', 'sec', 'red', 'cure']
  - 'stored': Near-matches = ['red', 'store', 'tor', 'ore']
  - 'website': Near-matches = ['sit', 'si', 'site', 'web']
--------------------------------------------------


--- Checking Seed Word Survival in Dictionary ---
['database', 'depends', 'encrypted', 'filesystems', 'internet', 'processing', 'restored', 'scheduling', 'secured', 'stored', 'website']
---

Validate Seed Words and Prepare ETA

In [13]:
logging.info("Started validating seed words")
seed_word_ids = {
    topic: [dictionary.token2id[word] for word in words if word in dictionary.token2id]
    for topic, words in seed_words.items()
}
for topic, ids in seed_word_ids.items():
    logging.info(f"Topic {topic}: {len(ids)}/{len(seed_words[topic])} seed words in dictionary")
    if not ids:
        logging.warning(f"No seed words for {topic} found in dictionary. Topic may be misaligned.")
logging.info("Completed validating seed words")

topic_name_to_id = {name: idx for idx, name in enumerate(seed_words.keys())}
print("\nTopic Name to Integer ID Mapping:")
print(topic_name_to_id)

num_topics = len(seed_words)
num_words = len(dictionary)
eta = np.ones((num_topics, num_words)) * 0.01
for topic_name, seed_word_ids_in_topic in seed_word_ids.items():
    topic_id = topic_name_to_id[topic_name]
    for word_id in seed_word_ids_in_topic:
        eta[topic_id][word_id] = 500.0


Topic Name to Integer ID Mapping:
{'F': 0, 'FT': 1, 'PE': 2, 'PO': 3, 'SC': 4, 'SE': 5, 'US': 6}


Train LDA Model

In [ ]:
logging.info("Started creating BoW corpus")
corpus = [dictionary.doc2bow(text) for text in tokenized_reviews]
logging.info("Completed creating BoW corpus")

logging.info("Started training LDA model")
print("\nTraining LDA model...")
lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    passes=20,
    alpha=0.01/num_topics,
    iterations=400,
    eta=eta,
    random_state=42,
    minimum_probability=0.05,
    per_word_topics=True,
    decay=0.7,
    offset=50.0
)
logging.info("Completed training LDA model")
print("Training complete.")


Training LDA model...
Training complete.


Save Models

In [15]:
logging.info("Saving LDA model, dictionary, and bigram model")
lda_model.save('models/lda_model')
dictionary.save('models/dictionary')
bigram_phraser.save('models/bigram_phraser')
logging.info("Saved LDA model, dictionary, and bigram models")

Analyze Topics and Coherence

In [16]:
logging.info("Computing per-topic coherence scores")
print("\n--- Per-Topic Coherence Scores ---")
coherence_model = CoherenceModel(
    model=lda_model,
    texts=tokenized_reviews,
    dictionary=dictionary,
    coherence='c_v',
    topn=10,
    window_size=50
)
per_topic_coherence = coherence_model.get_coherence_per_topic()
id_to_topic_name = {v: k for k, v in topic_name_to_id.items()}
for i, score in enumerate(per_topic_coherence):
    topic_name = id_to_topic_name[i]
    logging.info(f"Topic #{i} ({topic_name}): C_v Score = {score:.4f}")
    print(f"Topic #{i}: {topic_name} - C_v Score = {score:.4f}")
print("------------------------------------\n")

logging.info("Discovered Topics (Top 30 words):")
print("\n--- Deeper Dive into Discovered Topics (Top 30 words) ---")
for i in range(num_topics):
    topic_name = id_to_topic_name[i]
    topic_words_probs = lda_model.show_topic(i, topn=30)
    topic_words = [word for word, prob in topic_words_probs]
    log_message = f"Topic #{i} ({topic_name}): {', '.join(topic_words)}"
    logging.info(log_message)
    print(f"\nTopic #{i}: {topic_name}")
    print(topic_words)
print("----------------------------------------------------------\n")


--- Per-Topic Coherence Scores ---
Topic #0: F - C_v Score = 0.5489
Topic #1: FT - C_v Score = 0.4220
Topic #2: PE - C_v Score = 0.6918
Topic #3: PO - C_v Score = 0.3971
Topic #4: SC - C_v Score = 0.5677
Topic #5: SE - C_v Score = 0.5195
Topic #6: US - C_v Score = 0.3764
------------------------------------


--- Deeper Dive into Discovered Topics (Top 30 words) ---

Topic #0: F
['know', 'learned', 'better', 'program', 'need', 'lab', 'feel', 'taken', 'got', 'already', 'student', 'module', 'thought', 'section', 'basic', 'never', 'stuff', 'research', 'order', 'view', 'didnt', 'lesson', 'staff', 'change', 'knew', 'food', 'tough', 'understanding', 'art', 'short']

Topic #1: FT
['amazing', 'best', 'learned', 'experience', 'awesome', 'teaching', 'teacher', 'wonderful', 'instructor', 'excellent', 'love', 'learnt', 'professor', 'excel', 'ever', 'python', 'chuck', 'continue', 'team', 'opportunity', 'university', 'sir', 'fantastic', 'taught', 'prof', 'everything', 'happy', 'fun', 'hope', 'far']

Verify Seed Word Probabilities (Detailed)

In [17]:
print("\n--- Verifying Seed Word Probabilities and Ranks in Final Model ---")
logging.info("Verifying Seed Word Probabilities and Ranks in Final Model")
for topic_name, words in seed_words.items():
    topic_id = topic_name_to_id[topic_name]
    print(f"\nAssigned Topic: {topic_name} (ID: {topic_id})")
    logging.info(f"Assigned Topic: {topic_name} (ID: {topic_id})")
    
    for word in words:
        if word in dictionary.token2id:
            word_id = dictionary.token2id[word]
            term_topics = lda_model.get_term_topics(word_id, minimum_probability=0.0)
            print(f"  - Seed Word '{word}'")
            logging.info(f"  - Seed Word '{word}'")
            for t_id, prob in term_topics:
                t_name = id_to_topic_name[t_id]
                topic_words_probs = lda_model.show_topic(t_id, topn=len(dictionary))
                word_to_rank = {w: idx + 1 for idx, (w, _) in enumerate(topic_words_probs)}
                rank = word_to_rank.get(word, "N/A")
                log_message = f"    * Topic {t_name} (ID: {t_id}): Probability = {prob:.4f}, Rank = {rank}"
                logging.info(log_message)
                print(log_message)
        else:
            logging.warning(f"Seed Word '{word}' for topic '{topic_name}' was not in the final dictionary.")
            print(f"  - '{word}' (Not in dictionary)")
print("----------------------------------------------------------\n")


--- Verifying Seed Word Probabilities and Ranks in Final Model ---

Assigned Topic: F (ID: 0)
  - Seed Word 'player'
    * Topic F (ID: 0): Probability = 0.0024, Rank = 105
  - Seed Word 'display'
    * Topic F (ID: 0): Probability = 0.0026, Rank = 96
  - Seed Word 'meeting'
    * Topic F (ID: 0): Probability = 0.0027, Rank = 88
  - Seed Word 'dispute'
    * Topic F (ID: 0): Probability = 0.0024, Rank = 103
  - Seed Word 'program'
    * Topic F (ID: 0): Probability = 0.0126, Rank = 4
    * Topic SE (ID: 5): Probability = 0.0013, Rank = 165
  - Seed Word 'clinical'
    * Topic F (ID: 0): Probability = 0.0053, Rank = 32
  - Seed Word 'member'
    * Topic F (ID: 0): Probability = 0.0028, Rank = 84
  - Seed Word 'staff'
    * Topic F (ID: 0): Probability = 0.0064, Rank = 23
  - Seed Word 'student'
    * Topic F (ID: 0): Probability = 0.0095, Rank = 11
    * Topic PE (ID: 2): Probability = 0.0013, Rank = 194
    * Topic SC (ID: 4): Probability = 0.0021, Rank = 94
    * Topic SE (ID: 5): Pr

Compute Coherence Scores

In [18]:
logging.info("Started computing c_v coherence scores")
window_size = 50

print("\n--- Computing C_v Coherence ---")
logging.info("Computing c_v coherence")

coherence_model = CoherenceModel(
    model=lda_model,
    texts=tokenized_reviews,
    dictionary=dictionary,
    coherence='c_v',
    topn=10,
    window_size=window_size
)

per_topic_coherence = coherence_model.get_coherence_per_topic()
print("\nPer-Topic C_v Scores:")
for i, score in enumerate(per_topic_coherence):
    topic_name = id_to_topic_name[i]
    logging.info(f"Topic #{i} ({topic_name}): C_v Score = {score:.4f}")
    print(f"Topic #{i}: {topic_name} - C_v Score = {score:.4f}")

coherence_score = coherence_model.get_coherence()
logging.info(f"Overall C_v Coherence Score: {coherence_score:.4f}")
print(f"\nOverall C_v Coherence Score: {coherence_score:.4f}")
print("------------------------------------\n")


--- Computing C_v Coherence ---

Per-Topic C_v Scores:
Topic #0: F - C_v Score = 0.5489
Topic #1: FT - C_v Score = 0.4220
Topic #2: PE - C_v Score = 0.6918
Topic #3: PO - C_v Score = 0.3971
Topic #4: SC - C_v Score = 0.5677
Topic #5: SE - C_v Score = 0.5195
Topic #6: US - C_v Score = 0.3764

Overall C_v Coherence Score: 0.5033
------------------------------------



In [19]:
def compute_coherence_range(lda_model, tokenized_reviews, dictionary, window_size=50, topn_range=(10, 100, 10)):
    """
    Compute C_v coherence scores for a range of topn values.
    
    Parameters:
    - lda_model: Trained LDA model
    - tokenized_reviews: List of tokenized texts
    - dictionary: Gensim dictionary
    - window_size: Window size for coherence calculation
    - topn_range: Tuple of (start, end, step) for topn values
    """
    logging.info("Started computing C_v coherence scores for multiple topn values")
    print("\n--- Computing C_v Coherence for Multiple topn Values ---")
    
    start, end, step = topn_range
    coherence_results = []
    
    for topn in range(start, end + 1, step):
        logging.info(f"Computing C_v coherence for topn={topn}")
        print(f"\nComputing C_v Coherence for topn={topn}")
        
        coherence_model = CoherenceModel(
            model=lda_model,
            texts=tokenized_reviews,
            dictionary=dictionary,
            coherence='c_v',
            topn=topn,
            window_size=window_size
        )
        
        per_topic_coherence = coherence_model.get_coherence_per_topic()
        print(f"\nPer-Topic C_v Scores (topn={topn}):")
        for i, score in enumerate(per_topic_coherence):
            topic_name = id_to_topic_name[i]
            logging.info(f"Topic #{i} ({topic_name}): C_v Score = {score:.4f} (topn={topn})")
            print(f"Topic #{i}: {topic_name} - C_v Score = {score:.4f}")
        
        coherence_score = coherence_model.get_coherence()
        logging.info(f"Overall C_v Coherence Score (topn={topn}): {coherence_score:.4f}")
        print(f"\nOverall C_v Coherence Score (topn={topn}): {coherence_score:.4f}")
        print("-" * 35)
        
        coherence_results.append({
            'topn': topn,
            'per_topic_coherence': per_topic_coherence,
            'overall_coherence': coherence_score
        })
    
    logging.info("Completed computing C_v coherence scores for multiple topn values")
    print("\n--- Completed Computing C_v Coherence for Multiple topn Values ---")
    return coherence_results

# Call the new function
coherence_results = compute_coherence_range(
    lda_model=lda_model,
    tokenized_reviews=tokenized_reviews,
    dictionary=dictionary,
    window_size=50,
    topn_range=(10, 100, 10)
)


--- Computing C_v Coherence for Multiple topn Values ---

Computing C_v Coherence for topn=10

Per-Topic C_v Scores (topn=10):
Topic #0: F - C_v Score = 0.5489
Topic #1: FT - C_v Score = 0.4220
Topic #2: PE - C_v Score = 0.6918
Topic #3: PO - C_v Score = 0.3971
Topic #4: SC - C_v Score = 0.5677
Topic #5: SE - C_v Score = 0.5195
Topic #6: US - C_v Score = 0.3764

Overall C_v Coherence Score (topn=10): 0.5033
-----------------------------------

Computing C_v Coherence for topn=20

Per-Topic C_v Scores (topn=20):
Topic #0: F - C_v Score = 0.5759
Topic #1: FT - C_v Score = 0.3827
Topic #2: PE - C_v Score = 0.7421
Topic #3: PO - C_v Score = 0.3754
Topic #4: SC - C_v Score = 0.5097
Topic #5: SE - C_v Score = 0.4747
Topic #6: US - C_v Score = 0.3882

Overall C_v Coherence Score (topn=20): 0.4927
-----------------------------------

Computing C_v Coherence for topn=30

Per-Topic C_v Scores (topn=30):
Topic #0: F - C_v Score = 0.5401
Topic #1: FT - C_v Score = 0.4189
Topic #2: PE - C_v Score 

Assign Topics to Reviews and Save with Topic Probabilities

In [20]:
logging.info("Started getting topic distributions")
doc_topics = [lda_model.get_document_topics(doc, minimum_probability=0.0) for doc in corpus]
topic_matrix = np.zeros((len(corpus), num_topics))
for i, topics in enumerate(doc_topics):
    for topic_id, prob in topics:
        topic_matrix[i, topic_id] = prob
logging.info("Completed getting topic distributions")

logging.info("Started saving all reviews with topics")
all_topics_df = filtered_df.copy()
all_topics_df['Topic'] = np.argmax(topic_matrix, axis=1)
all_topics_df['topic_name'] = all_topics_df['Topic'].map({v: k for k, v in topic_name_to_id.items()})
all_topics_path = '../../datasets/reviews_with_topic.csv'
all_topics_df[['processed_reviews', 'Topic', 'topic_name']].to_csv(all_topics_path, index=False)
logging.info(f"All reviews with topic assignments saved to {all_topics_path}")
print(f"\nAll reviews with topic assignments saved to {all_topics_path}")

logging.info("Sample of saved data:")
logging.info(all_topics_df[['processed_reviews', 'Topic', 'topic_name']].head(2).to_string())
print("Sample of saved data:")
print(all_topics_df[['processed_reviews', 'Topic', 'topic_name']].head(2))


All reviews with topic assignments saved to ../../datasets/reviews_with_topic.csv
Sample of saved data:
                                   processed_reviews  Topic topic_name
0  thank much lot good find course like especiall...      4         SC
1  reading many negative true certain becomes wai...      2         PE


Select and Save Pseudo-Labeled Reviews with Topic Probabilities and Entropy Filtering

In [ ]:
logging.info(f"Started selecting reviews with confidence > {threshold} and entropy <= {entropy_threshold}")
selected_reviews = {topic_name: [] for topic_name in seed_words.keys()}
non_selected_reviews = {topic_name: [] for topic_name in seed_words.keys()}
id_to_topic = {v: k for k, v in topic_name_to_id.items()}

# Compute topic entropy for all reviews
entropies = [entropy(probs) for probs in topic_matrix]
logging.info(f"Average topic entropy: {np.mean(entropies):.4f}, Std: {np.std(entropies):.4f}")
print(f"\nAverage topic entropy: {np.mean(entropies):.4f}, Std: {np.std(entropies):.4f}")

for i, (review_text, topic_probs) in enumerate(zip(filtered_df['processed_reviews'], topic_matrix)):
    dominant_topic = np.argmax(topic_probs)
    dominant_prob = topic_probs[dominant_topic]
    topic_name = id_to_topic[dominant_topic]
    review_entropy = entropy(topic_probs)
    review_data = {
        'original_index': valid_indices[i],
        'topic': topic_name,
        'confidence': dominant_prob,
        'text': review_text,
        'topic_probs': ','.join(map(str, topic_probs))
    }
    if dominant_prob > threshold and review_entropy <= entropy_threshold:
        selected_reviews[topic_name].append(review_data)
    else:
        non_selected_reviews[topic_name].append(review_data)
logging.info("Completed selecting reviews")

# Log and print summary
logging.info("High-Confidence Reviews Summary:")
print("\nHigh-Confidence Reviews Summary:")
for topic_name, reviews in selected_reviews.items():
    logging.info(f"Topic {topic_name}: {len(reviews):,} high-confidence reviews selected")
    print(f"Topic {topic_name}: {len(reviews):,} high-confidence reviews selected")

logging.info("Non-High-Confidence Reviews Summary:")
print("\nNon-High-Confidence Reviews Summary:")
for topic_name, reviews in non_selected_reviews.items():
    logging.info(f"Topic {topic_name}: {len(reviews):,} non-high-confidence reviews selected")
    print(f"Topic {topic_name}: {len(reviews):,} non-high-confidence reviews selected")

# Save high-confidence reviews
logging.info("Started saving high-confidence reviews")
selected_data = []
for topic_name, reviews in selected_reviews.items():
    for rev in reviews:
        selected_data.append({
            'original_index': rev['original_index'],
            'topic': rev['topic'],
            'confidence': rev['confidence'],
            'text': rev['text'],
            'topic_probs': rev['topic_probs']
        })
selected_df = pd.DataFrame(selected_data)
selected_df.to_csv(output_path, index=False)
logging.info(f"High-confidence pseudo-labeled reviews saved to {output_path}")
print(f"\nHigh-confidence pseudo-labeled reviews saved to {output_path}")

# Save non-high-confidence reviews
logging.info("Started saving non-high-confidence reviews")
non_selected_data = []
for topic_name, reviews in non_selected_reviews.items():
    for rev in reviews:
        non_selected_data.append({
            'original_index': rev['original_index'],
            'text': rev['text']
        })
non_selected_df = pd.DataFrame(non_selected_data)
non_selected_path = '../../datasets/non_high_confidence_reviews.csv'
non_selected_df.to_csv(non_selected_path, index=False)
logging.info(f"Non-high-confidence reviews saved to {non_selected_path}")
print(f"\nNon-high-confidence reviews saved to {non_selected_path}")


Average topic entropy: 0.6271, Std: 0.4455

High-Confidence Reviews Summary:
Topic F: 4,688 high-confidence reviews selected
Topic FT: 17,337 high-confidence reviews selected
Topic PE: 16,161 high-confidence reviews selected
Topic PO: 6,126 high-confidence reviews selected
Topic SC: 19,792 high-confidence reviews selected
Topic SE: 16,516 high-confidence reviews selected
Topic US: 12,495 high-confidence reviews selected

Non-High-Confidence Reviews Summary:
Topic F: 14,527 non-high-confidence reviews selected
Topic FT: 25,572 non-high-confidence reviews selected
Topic PE: 43,897 non-high-confidence reviews selected
Topic PO: 10,594 non-high-confidence reviews selected
Topic SC: 40,842 non-high-confidence reviews selected
Topic SE: 32,614 non-high-confidence reviews selected
Topic US: 20,366 non-high-confidence reviews selected

High-confidence pseudo-labeled reviews saved to ../../datasets/selected_pseudo_labeled.csv

Non-high-confidence reviews saved to ../../datasets/non_high_confid